# Validação 12 — Síntese das evidências

## Goal

Comprovar que as relações são resumidas primeiro por artigo e somente depois entre artigos, mantendo direção, força, qualidade e conflitos como dimensões separadas.

## Setup

Trechos do mesmo artigo são correlacionados e não representam estudos independentes. Por isso, suas probabilidades são médias dentro do artigo; cada artigo entra apenas uma vez na síntese final. O agregador recebe um perfil explícito de qualidade em vez de tentar inferi-lo silenciosamente. Como risco de viés, protocolo e retratação ainda não foram validados, o perfil deste exemplo permanece `UNCLEAR`.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_NLI_MODEL,
    ArticleQualityProfile,
    Bm25Index,
    ChunkingConfig,
    ClassificationConfig,
    EvidenceStrength,
    ExtractionConfig,
    HybridIndex,
    PmcClient,
    PubMedClient,
    QualityLevel,
    SemanticIndex,
    SentenceTransformerEncoder,
    TransformersNliClassifier,
    build_claim_evidence_pairs,
    chunk_article_content,
    classify_claim_evidence_pairs,
    extract_evidence_statements,
    prepare_search_plan,
    retrieve_article_content,
    search_pubmed,
    synthesize_evidence,
    validate_analysis_input,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f'Execução UTC: {executed_at}')

Execução UTC: 2026-09-25T11:49:40.457367+00:00


## Steps

### 1. Reexecutar o pipeline até as relações

O artigo real é recuperado do PubMed/PMC, passa pelos rankings lexical e semântico, pela extração e pelo classificador NLI multilíngue.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ['33431520[pmid]']

claim = 'Beber café pode alterar o risco de câncer de próstata.'
analysis_input = validate_analysis_input(claim)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_result = search_pubmed(
    search_plan,
    PubMedClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
    max_results_per_query=1,
)
publication = pubmed_result.publications[0]
content = retrieve_article_content(
    publication,
    PmcClient(email=os.getenv('NCBI_EMAIL'), api_key=os.getenv('NCBI_API_KEY')),
)
chunks = chunk_article_content(content, ChunkingConfig(max_words=120, overlap_words=20))
lexical_index = Bm25Index(chunks)
semantic_index = SemanticIndex(
    chunks,
    SentenceTransformerEncoder(os.getenv('EMBEDDING_MODEL', DEFAULT_EMBEDDING_MODEL)),
)
hybrid_results = HybridIndex(lexical_index, semantic_index).search(claim, top_k=8)
statements = extract_evidence_statements(
    claim,
    hybrid_results,
    ExtractionConfig(max_statements=6, max_per_chunk=2, min_words=8, max_words=80),
)
pairs = build_claim_evidence_pairs(claim, statements)
classifier = TransformersNliClassifier(os.getenv('NLI_MODEL', DEFAULT_NLI_MODEL))
assessments = classify_claim_evidence_pairs(
    pairs,
    classifier,
    ClassificationConfig(minimum_confidence=0.60, minimum_margin=0.10),
)
print(f'Artigo: {publication.title}')
print(f'PMID: {publication.pmid} | relações: {len(assessments)}')

/private/tmp/fatofake-notebook-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22282.14it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 16370.60it/s]

Artigo: Coffee consumption and risk of prostate cancer: a systematic review and meta-analysis.
PMID: 33431520 | relações: 6


### 2. Declarar o que sabemos sobre qualidade

O título do PubMed identifica uma revisão sistemática e meta-análise, mas o desenho declarado não basta para graduar a qualidade. Mantemos `UNCLEAR` e registramos exatamente o que falta verificar.

In [3]:
assert 'systematic review and meta-analysis' in publication.title.casefold()
quality_profile = ArticleQualityProfile(
    pmid=publication.pmid,
    level=QualityLevel.UNCLEAR,
    study_design='Systematic review and meta-analysis (declarado no título)',
    rationale=(
        'Risco de viés, protocolo, retratação e disponibilidade dos dados ainda '
        'não foram verificados por módulos especializados.'
    ),
    source_url=publication.url,
)
pprint({
    'pmid': quality_profile.pmid,
    'study_design': quality_profile.study_design,
    'quality': quality_profile.level.value,
    'quality_weight': quality_profile.weight,
    'rationale': quality_profile.rationale,
    'source_url': quality_profile.source_url,
})

{'pmid': '33431520',
 'quality': 'UNCLEAR',
 'quality_weight': 0.25,
 'rationale': 'Risco de viés, protocolo, retratação e disponibilidade dos '
              'dados ainda não foram verificados por módulos especializados.',
 'source_url': 'https://pubmed.ncbi.nlm.nih.gov/33431520/',
 'study_design': 'Systematic review and meta-analysis (declarado no título)'}


### 3. Sintetizar por artigo e por conjunto

A direção é um sinal probabilístico. A força responde a outra pergunta: há estudos independentes e qualidade suficiente para sustentar uma conclusão?

In [4]:
synthesis = synthesize_evidence(assessments, [quality_profile])
article = synthesis.articles[0]

pprint({
    'directional_signal': synthesis.direction.value,
    'evidence_strength': synthesis.strength.value,
    'article_count': synthesis.article_count,
    'has_conflict': synthesis.has_conflict,
    'p_support': round(synthesis.support_probability, 4),
    'p_contradiction': round(synthesis.contradiction_probability, 4),
    'p_neutral': round(synthesis.neutral_probability, 4),
    'rationale': synthesis.rationale,
})
print()
pprint({
    'pmid': article.pmid,
    'relations': dict(article.relation_counts),
    'uncertain_relations': article.uncertain_count,
    'internal_conflict': article.has_internal_conflict,
    'quality': article.quality.level.value,
})

{'article_count': 1,
 'directional_signal': 'SUPPORTS',
 'evidence_strength': 'INSUFFICIENT',
 'has_conflict': False,
 'p_contradiction': 0.0381,
 'p_neutral': 0.3463,
 'p_support': 0.6156,
 'rationale': 'Há um sinal de apoio, mas apenas 1 artigo independente; são '
              'necessários ao menos 2.'}

{'internal_conflict': False,
 'pmid': '33431520',
 'quality': 'UNCLEAR',
 'relations': {'NEUTRAL': 1, 'SUPPORTS': 4, 'UNCERTAIN': 1},
 'uncertain_relations': 1}


## Checks

As verificações confirmam normalização, proveniência, insuficiência com artigo único e ausência de multiplicação artificial do peso ao repetir trechos do mesmo artigo.

In [5]:
assert synthesis.article_count == 1
assert synthesis.strength is EvidenceStrength.INSUFFICIENT
assert len(synthesis.articles) == 1
assert synthesis.articles[0].assessment_count == len(assessments)
assert synthesis.articles[0].quality.level is QualityLevel.UNCLEAR
assert abs(
    synthesis.support_probability
    + synthesis.contradiction_probability
    + synthesis.neutral_probability
    - 1.0
) < 1e-6

duplicated = synthesize_evidence([*assessments, *assessments], [quality_profile])
assert duplicated.article_count == 1
assert abs(duplicated.support_probability - synthesis.support_probability) < 1e-12
assert abs(duplicated.contradiction_probability - synthesis.contradiction_probability) < 1e-12
assert abs(duplicated.neutral_probability - synthesis.neutral_probability) < 1e-12

print(
    'Validação aprovada: o sinal direcional foi preservado, mas um único artigo '
    'com qualidade ainda não avaliada permaneceu como evidência insuficiente.'
)

Validação aprovada: o sinal direcional foi preservado, mas um único artigo com qualidade ainda não avaliada permaneceu como evidência insuficiente.


## Next Steps

A síntese estará validada quando todas as células forem executadas sem erros. A próxima etapa será preencher os perfis de qualidade por meio de validações externas de retratação, protocolo, desenho do estudo e disponibilidade dos dados.